# 02  -  Feature Matrix Construction

**Goal:** Convert raw tool outputs into a single tidy feature matrix (`data/processed/feature_matrix.parquet`).

**Construction order:**
1. DefenseFinder (defence + anti-defence systems)
2. PADLOC (defence systems)
3. Merge DF + PADLOC via `system_name_map.csv`
4. ResFinder (ARG counts)
5. ICEberg (IME/ICE counts, with BLAST coverage filter)
6. BacMet (HMRG counts, SUPERSEDED  -  see 6b)
7. AMRFinderPlus (metal resistance genes)
8. ISEScan (IS element counts by family)
9. MLST + metadata
10. Derived features: ratios, sparsity filter, ARG burden tertile labels
11. Save to parquet

## Imports and paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Project layout ──────────────────────────────────────────────────────────
ROOT    = Path("..")                   # project root (one level up from notebooks/)
INTERIM = ROOT / "data" / "interim"   # per-species tool outputs
PROC    = ROOT / "data" / "processed" # final feature matrix lands here
CONFIG  = ROOT / "config"

PROC.mkdir(parents=True, exist_ok=True)  # create processed/ if missing

# ── Six ESKAPE species, matching interim/ subdirectory names ─────────────────
SPECIES = [
    "abaumannii",
    "ecloaceae",
    "efaecium",
    "kpneumoniae",
    "paeruginosa",
    "saureus",
]

# ── Accession normalisation ───────────────────────────────────────────────
def norm_acc(s: str) -> str:
    """GCF_000505685_1 → GCF_000505685.1"""
    prefix, version = s.rsplit("_", 1)   # split on LAST underscore only
    return f"{prefix}.{version}"

print("Paths OK")
print(f"  interim : {INTERIM.resolve()}")
print(f"  processed: {PROC.resolve()}")

Paths OK
  interim : /Users/Vicky/Acinetobacter_ML_2/eskape-defence-ml_2/data/interim
  processed: /Users/Vicky/Acinetobacter_ML_2/eskape-defence-ml_2/data/processed


## Section 1  -  Parse DefenseFinder outputs

In [2]:
# ── Load system name map ─────────────────────────────────────────────────────
name_map = pd.read_csv(CONFIG / "system_name_map.csv")

print(f"system_name_map: {len(name_map)} rows")
print(name_map["source"].value_counts().to_string())
print()

# Build lookup: df_subtype → canonical_name
_df_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["df_subtype"])
df_subtype_to_canonical = _df_map_rows.set_index("df_subtype")["canonical_name"].to_dict()

print(f"DF subtype → canonical lookup: {len(df_subtype_to_canonical)} entries")
# Spot-check the key systems from the published paper
for key in ["SspBCDE", "Gao_Qat", "RM_Type_I", "BREX_I", "NARP1"]:
    print(f"  {key!r:20s} → {df_subtype_to_canonical.get(key, 'MISSING')}")

system_name_map: 448 rows
source
df_only           150
padloc_only       144
both              113
df_antidefense     40
exclude             1

DF subtype → canonical lookup: 303 entries
  'SspBCDE'            → SspBCDE
  'Gao_Qat'            → Gao_Qat
  'RM_Type_I'          → RM_Type_I
  'BREX_I'             → BREX_I
  'NARP1'              → adf_NARP1


### Parse defence systems (activity == "Defense")

In [3]:
df_defense_records = []   # accumulates one mini-dataframe per genome
df_missing = []           # tracks missing TSV files (should be zero)

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"  # e.g. data/interim/abaumannii/defensefinder/

    for genome_dir in sorted(df_dir.iterdir()):  # each subdirectory = one genome
        if not genome_dir.is_dir():
            continue  # skip .DS_Store and other non-directory files

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            df_missing.append((sp, genome_dir.name))  # log, don't crash
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue  # genome has zero defence systems  -  valid result

        defense_rows = raw[raw["activity"] == "Defense"][["subtype"]].copy()

        if defense_rows.empty:
            continue

        # Tag with canonical genome_id and species
        defense_rows["genome_id"] = norm_acc(genome_dir.name)  # GCF_000505685_1 → GCF_000505685.1
        defense_rows["species"]   = sp

        df_defense_records.append(defense_rows)

# ── Concatenate all species ─────────────────────────────────────────────────
if df_defense_records:
    df_defense_long = pd.concat(df_defense_records, ignore_index=True)
else:
    df_defense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

print(f"DefenseFinder defence rows (raw, before name mapping): {len(df_defense_long):,}")
print(f"Missing TSV files: {len(df_missing)}")
if df_missing:
    print("  ", df_missing[:10])

DefenseFinder defence rows (raw, before name mapping): 30,556
Missing TSV files: 0


In [4]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
df_defense_long["canonical_name"] = df_defense_long["subtype"].map(df_subtype_to_canonical)

unmapped = df_defense_long[df_defense_long["canonical_name"].isna()]
if not unmapped.empty:
    print(f"WARNING: {len(unmapped)} rows have unmapped subtypes:")
    print(unmapped["subtype"].value_counts().head(20).to_string())
else:
    print("All DF defence subtypes mapped successfully.")

# Drop unmapped rows
df_defense_long = df_defense_long.dropna(subset=["canonical_name"])

print(f"\nRows after mapping: {len(df_defense_long):,}")
print(f"Unique canonical systems detected: {df_defense_long['canonical_name'].nunique()}")
print(f"Unique genomes with >=1 defence system: {df_defense_long['genome_id'].nunique()}")
print()
print("Top 10 most frequent defence systems:")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

subtype
DS-8                 9
Gao_Ape              8
gcuWGS21             7
DS-11                5
VP1796               5
HEC-06               4
AbiV                 4
DS-27                4
UG5_small            4
CAS_Class1-Type-I    4
DS-43                3
DS-44                3
HEC-03               3
Retron_XII           3
Gao_Her_SIR          2
VP1826               2
PD-T4-10             2
Retron_VII_2         2
pAgo_S1A             1
AbiO                 1

Rows after mapping: 30,471
Unique canonical systems detected: 263
Unique genomes with >=1 defence system: 3450

Top 10 most frequent defence systems:
canonical_name
RM_Type_I         3479
RM_Type_IV        1313
RM_Type_II        1225
Gabija             904
AbiE               806
RosmerTA           776
df_gcu233          771
df_Mok_Hok_Sok     675
df_MazEF           640
df_FS_Sma          577


### Parse anti-defence systems (activity == "Antidefense")

In [5]:
df_antidefense_records = []

for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"

    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        tsv_path = genome_dir / "defense_finder_systems.tsv"

        if not tsv_path.exists():
            continue

        raw = pd.read_csv(tsv_path, sep="\t")

        if raw.empty:
            continue

        antidefense_rows = raw[raw["activity"] == "Antidefense"][["subtype"]].copy()

        if antidefense_rows.empty:
            continue

        antidefense_rows["genome_id"] = norm_acc(genome_dir.name)
        antidefense_rows["species"]   = sp

        df_antidefense_records.append(antidefense_rows)

if df_antidefense_records:
    df_antidefense_long = pd.concat(df_antidefense_records, ignore_index=True)
else:
    df_antidefense_long = pd.DataFrame(columns=["subtype", "genome_id", "species"])

# Map to canonical names
df_antidefense_long["canonical_name"] = df_antidefense_long["subtype"].map(df_subtype_to_canonical)

unmapped_adf = df_antidefense_long[df_antidefense_long["canonical_name"].isna()]
if not unmapped_adf.empty:
    print(f"WARNING: {len(unmapped_adf)} unmapped anti-defence subtypes:")
    print(unmapped_adf["subtype"].value_counts().head(10).to_string())

df_antidefense_long = df_antidefense_long.dropna(subset=["canonical_name"])

print(f"Anti-defence rows: {len(df_antidefense_long):,}")
print(f"Unique anti-defence systems: {df_antidefense_long['canonical_name'].nunique()}")
print(f"Genomes carrying >=1 anti-defence system: {df_antidefense_long['genome_id'].nunique()}")
print()
print("Top anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(10).to_string())

subtype
acrif1             10
dar_ddr_hdf_ulx     3
gnarl1              3
acrie6              3
acrie7              3
acric3              3
acric4              3
ddra                1
hdf                 1
gad1                1
Anti-defence rows: 11,085
Unique anti-defence systems: 40
Genomes carrying >=1 anti-defence system: 3032

Top anti-defence systems:
canonical_name
adf_ardb_klca_acric11    1977
adf_arda_ardu            1432
adf_acriia21             1343
adf_psiab                 866
adf_Aca_alone             742
adf_ardc                  641
adf_apyc1                 610
adf_NARP1                 570
adf_Adnd_p0020_p0021      454
adf_acrie9                373


### Section 1 outputs

`df_defense_long` and `df_antidefense_long` are long-form tables  -  one row per system instance per genome. Pivot is deferred to Section 3, where DF and PADLOC long-form tables are merged before pivoting to wide.

In [6]:
# Section 1 summary  -  print shapes of long-form tables
print(f"df_defense_long    : {len(df_defense_long):,} rows | "
      f"{df_defense_long['genome_id'].nunique()} genomes | "
      f"{df_defense_long['canonical_name'].nunique()} unique systems")

print(f"df_antidefense_long: {len(df_antidefense_long):,} rows | "
      f"{df_antidefense_long['genome_id'].nunique()} genomes | "
      f"{df_antidefense_long['canonical_name'].nunique()} unique anti-defence systems")

print()
print("Top 10 defence systems (by instance count across all genomes):")
print(df_defense_long["canonical_name"].value_counts().head(10).to_string())

print()
print("Top 5 anti-defence systems:")
print(df_antidefense_long["canonical_name"].value_counts().head(5).to_string())

df_defense_long    : 30,471 rows | 3450 genomes | 263 unique systems
df_antidefense_long: 11,085 rows | 3032 genomes | 40 unique anti-defence systems

Top 10 defence systems (by instance count across all genomes):
canonical_name
RM_Type_I         3479
RM_Type_IV        1313
RM_Type_II        1225
Gabija             904
AbiE               806
RosmerTA           776
df_gcu233          771
df_Mok_Hok_Sok     675
df_MazEF           640
df_FS_Sma          577

Top 5 anti-defence systems:
canonical_name
adf_ardb_klca_acric11    1977
adf_arda_ardu            1432
adf_acriia21             1343
adf_psiab                 866
adf_Aca_alone             742


### Verify: coverage across all genomes

In [7]:
# All genome_ids seen across defence + antidefence
all_df_genomes = set(df_defense_long["genome_id"]) | set(df_antidefense_long["genome_id"])
print(f"Genomes with >=1 DF hit (defence or anti-defence): {len(all_df_genomes)}")

all_genome_ids = []
for sp in SPECIES:
    df_dir = INTERIM / sp / "defensefinder"
    for genome_dir in sorted(df_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store
        all_genome_ids.append(norm_acc(genome_dir.name))

all_genome_ids = sorted(set(all_genome_ids))
print(f"Total genome IDs from directory scan:               {len(all_genome_ids)}")

zero_hit_genomes = set(all_genome_ids) - all_df_genomes
print(f"Genomes with zero DF hits (pure zeros):             {len(zero_hit_genomes)}")

Genomes with >=1 DF hit (defence or anti-defence): 3452
Total genome IDs from directory scan:               3460
Genomes with zero DF hits (pure zeros):             8


## Section 2  -  Parse PADLOC outputs

In [8]:
# ── Build padloc_system → canonical_name lookup ──────────────────────────────
_padloc_map_rows = name_map[name_map["source"] != "exclude"].dropna(subset=["padloc_system"])
padloc_system_to_canonical = _padloc_map_rows.set_index("padloc_system")["canonical_name"].to_dict()

print(f"PADLOC system → canonical lookup: {len(padloc_system_to_canonical)} entries")
for key in ["AbiC", "PT_SspABCD", "qatABCD", "brex_type_I", "AVAST_type_II"]:
    print(f"  {key!r:22s} → {padloc_system_to_canonical.get(key, 'MISSING')}")

PADLOC system → canonical lookup: 257 entries
  'AbiC'                 → AbiC
  'PT_SspABCD'           → SspBCDE
  'qatABCD'              → Gao_Qat
  'brex_type_I'          → BREX_I
  'AVAST_type_II'        → AVAST_II


In [9]:
padloc_records = []
padloc_missing  = []   # files absent entirely
padloc_unmapped_subtypes = []  # system names not in name_map

for sp in SPECIES:
    padloc_dir = INTERIM / sp / "padloc"  # flat directory  -  one CSV per genome

    for csv_file in sorted(padloc_dir.iterdir()):
        if not csv_file.name.endswith("_padloc.csv"):
            continue

        # Reconstruct genome_id: strip the trailing '_padloc' before normalising
        stem = csv_file.stem.removesuffix("_padloc")   # GCF_000505685_1_padloc → GCF_000505685_1
        genome_id = norm_acc(stem)                       # GCF_000505685_1 → GCF_000505685.1

        if csv_file.stat().st_size == 0:
            continue  # 0-byte = PADLOC ran but found no hits

        raw = pd.read_csv(csv_file)

        if raw.empty:
            continue  # genome has zero PADLOC hits  -  valid

        # Deduplicate to one row per system INSTANCE:
        instances = raw.drop_duplicates(subset=["system", "system.number"])[["system"]].copy()

        if instances.empty:
            continue

        instances["genome_id"] = genome_id
        instances["species"]   = sp

        padloc_records.append(instances)

# ── Concatenate ──────────────────────────────────────────────────────────────
if padloc_records:
    padloc_long_raw = pd.concat(padloc_records, ignore_index=True)
else:
    padloc_long_raw = pd.DataFrame(columns=["system", "genome_id", "species"])

print(f"PADLOC rows after instance deduplication (raw): {len(padloc_long_raw):,}")
print(f"Unique PADLOC system names seen: {padloc_long_raw['system'].nunique()}")
print(f"Genomes with >=1 PADLOC hit: {padloc_long_raw['genome_id'].nunique()}")

PADLOC rows after instance deduplication (raw): 41,313
Unique PADLOC system names seen: 281
Genomes with >=1 PADLOC hit: 3456


In [10]:
# ── Apply canonical name mapping ─────────────────────────────────────────────
padloc_long_raw["canonical_name"] = padloc_long_raw["system"].map(padloc_system_to_canonical)

unmapped_padloc = padloc_long_raw[padloc_long_raw["canonical_name"].isna()]
if not unmapped_padloc.empty:
    print(f"WARNING: {len(unmapped_padloc)} rows with unmapped PADLOC system names:")
    print(unmapped_padloc["system"].value_counts().head(20).to_string())
    print()
else:
    print("All PADLOC system names mapped successfully.")

padloc_long = padloc_long_raw.dropna(subset=["canonical_name"]).copy()

print(f"Rows after mapping: {len(padloc_long):,}")
print(f"Unique canonical systems from PADLOC: {padloc_long['canonical_name'].nunique()}")
print(f"Genomes with >=1 mapped PADLOC system: {padloc_long['genome_id'].nunique()}")
print()
print("Top 10 PADLOC systems (by instance count):")
print(padloc_long["canonical_name"].value_counts().head(10).to_string())
print()

# ── Cross-tool sanity check ───────────────────────────────────────────────────
df_systems   = set(df_defense_long["canonical_name"].unique())
padloc_sys   = set(padloc_long["canonical_name"].unique())
in_both      = df_systems & padloc_sys
df_only_seen = df_systems - padloc_sys
padloc_only_seen = padloc_sys - df_systems

print(f"Systems seen by BOTH tools:    {len(in_both)}")
print(f"Systems seen by DF only:       {len(df_only_seen)}")
print(f"Systems seen by PADLOC only:   {len(padloc_only_seen)}")

system
DMS_other         3474
PDC-M02              5
argonaute_solo       4
AbiV                 3
PDC-M29              3
HEC-03               3
PDC-M35              2
PDC-M71              2
PDC-M54              2
PDC-M36              2
PD-T4-10             2
retron_V             2
thoeris_other        1
PDC-S63              1
retron_IX            1
PD-T4-9              1
PDC-M12              1
PDC-M69              1
PT_PbeABCD           1
wadjet_type_II       1

Rows after mapping: 37,797
Unique canonical systems from PADLOC: 257
Genomes with >=1 mapped PADLOC system: 3455

Top 10 PADLOC systems (by instance count):
canonical_name
padloc_PDC-S07    2740
RM_Type_I         2329
padloc_SoFic      1636
AbiE              1350
padloc_PDC-S04    1288
padloc_PDC-S12    1227
Gabija            1218
PD-T4-6           1183
VSPR              1182
RM_Type_II         872

Systems seen by BOTH tools:    113
Systems seen by DF only:       150
Systems seen by PADLOC only:   144


## Section 3  -  Merge DF + PADLOC → defence feature block

**Stage A:** count instances per (genome, system, tool).
**Stage B:** `presence` = union (either tool); `count` = max(df, padloc); `n_tools` = number of tools detecting it.
**Stage C:** pivot to wide, reindex against full genome list, sparsity filter (< 5 genomes → drop).

In [11]:
# ── Stage A: count instances per (genome, system, tool) ─────────────────────
# Tag each table with its source tool before concatenating
combined_long = pd.concat([
    df_defense_long.assign(tool="df"),
    padloc_long.assign(tool="padloc"),
], ignore_index=True)

# groupby gives us: for each (genome, system, tool), how many instances?
tool_counts = (
    combined_long
    .groupby(["genome_id", "canonical_name", "tool"])
    .size()                             # row count = instance count
    .reset_index(name="count")
)

# Pivot tool axis: index = (genome_id, canonical_name), columns = tool names
counts_by_tool = tool_counts.pivot_table(
    index=["genome_id", "canonical_name"],
    columns="tool",
    values="count",
    fill_value=0,       # 0 means that tool detected nothing for this (genome, system)
).reset_index()

counts_by_tool.columns.name = None
for col in ["df", "padloc"]:       # ensure both columns exist even if one tool had no hits
    if col not in counts_by_tool.columns:
        counts_by_tool[col] = 0

print(f"(genome, system) pairs detected by at least one tool: {len(counts_by_tool):,}")
print(f"Preview:")
print(counts_by_tool.head(6).to_string())

(genome, system) pairs detected by at least one tool: 47,391
Preview:
         genome_id  canonical_name   df  padloc
0  GCF_000006765.1          Gabija  1.0     0.0
1  GCF_000006765.1         PD-T4-6  0.0     1.0
2  GCF_000006765.1       RM_Type_I  1.0     1.0
3  GCF_000006765.1      Retron_I-B  0.0     1.0
4  GCF_000006765.1    Rst_Helicase  1.0     1.0
5  GCF_000006765.1  padloc_PDC-S39  0.0     1.0


In [12]:
# ── Stage B: compute presence, count, n_tools ────────────────────────────────
counts_by_tool["presence"] = 1   # every row here is a detected (genome, system) pair
counts_by_tool["count"]    = counts_by_tool[["df", "padloc"]].max(axis=1)
counts_by_tool["n_tools"]  = (
    (counts_by_tool["df"] > 0).astype(int) +
    (counts_by_tool["padloc"] > 0).astype(int)
)

print("n_tools distribution (how many tools agreed on each (genome, system) detection):")
print(counts_by_tool["n_tools"].value_counts().sort_index().to_string())
print()

# Spot-check: key systems from the published paper
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "AbiE"]:
    subset = counts_by_tool[counts_by_tool["canonical_name"] == sys]
    n2 = (subset["n_tools"] == 2).sum()
    n1 = (subset["n_tools"] == 1).sum()
    total = len(subset)
    print(f"  {sys:20s}: {total} genomes | both tools={n2} | one tool={n1}")

n_tools distribution (how many tools agreed on each (genome, system) detection):
n_tools
1    33649
2    13742

  SspBCDE             : 311 genomes | both tools=309 | one tool=2
  Gao_Qat             : 271 genomes | both tools=266 | one tool=5
  RM_Type_I           : 2265 genomes | both tools=1574 | one tool=691
  AbiE                : 1123 genomes | both tools=783 | one tool=340


In [13]:
# ── Stage C: pivot to wide, reindex, sparsity filter ─────────────────────────

# Pivot presence → genome × system binary matrix
defence_pa = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="presence")
    .fillna(0)          # genomes absent from this system's rows → 0
    .astype(int)
)

# Pivot count → genome × system count matrix
defence_count = (
    counts_by_tool
    .pivot(index="genome_id", columns="canonical_name", values="count")
    .fillna(0)
    .astype(int)
)

# Reindex both against the full 900-genome list
defence_pa    = defence_pa.reindex(all_genome_ids, fill_value=0)
defence_count = defence_count.reindex(all_genome_ids, fill_value=0)

print(f"Before sparsity filter: {defence_pa.shape[1]} defence systems")

# Sparsity filter: drop systems present in < 5 genomes
system_prevalence = defence_pa.sum(axis=0)   # number of genomes with presence=1
keep_systems = system_prevalence[system_prevalence >= 5].index

defence_pa    = defence_pa[keep_systems]
defence_count = defence_count[keep_systems]

print(f"After sparsity filter (>=5 genomes): {defence_pa.shape[1]} defence systems")
print(f"Systems dropped (too rare): {len(system_prevalence) - len(keep_systems)}")
print(f"Defence P/A matrix shape: {defence_pa.shape}")
print()
print("Sparsity of final defence P/A matrix:")
print(f"  {(defence_pa == 0).values.mean():.1%}")

Before sparsity filter: 407 defence systems
After sparsity filter (>=5 genomes): 360 defence systems
Systems dropped (too rare): 47
Defence P/A matrix shape: (3460, 360)

Sparsity of final defence P/A matrix:
  96.2%


In [14]:
# ── Anti-defence: straight pivot (DF only, no merge needed) ─────────────────
adef_pa = (
    df_antidefense_long
    .groupby(["genome_id", "canonical_name"])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)                       # presence/absence
    .reindex(all_genome_ids, fill_value=0)
)

# Sparsity filter  -  same threshold
adef_prevalence = adef_pa.sum(axis=0)
keep_adef = adef_prevalence[adef_prevalence >= 5].index
adef_pa = adef_pa[keep_adef]

print(f"Anti-defence P/A matrix: {adef_pa.shape}  (after sparsity filter)")
print(f"Anti-defence systems dropped: {len(adef_prevalence) - len(keep_adef)}")
print()

# ── Summary of defence feature block ─────────────────────────────────────────
print("=" * 55)
print("DEFENCE FEATURE BLOCK SUMMARY")
print("=" * 55)
print(f"Genomes (rows)         : {defence_pa.shape[0]}")
print(f"Defence systems (cols) : {defence_pa.shape[1]}")
print(f"Anti-defence (cols)    : {adef_pa.shape[1]}")
print(f"Total feature columns  : {defence_pa.shape[1] + adef_pa.shape[1]}")
print()
# Verify key systems from published paper survived the filter
for sys in ["SspBCDE", "Gao_Qat", "RM_Type_I", "RM_Type_II", "RM_Type_IV"]:
    status = "PRESENT" if sys in defence_pa.columns else "DROPPED"
    if status == "PRESENT":
        prev = int(defence_pa[sys].sum())
        print(f"  {sys:20s}: {status} ({prev}/900 genomes)")
    else:
        print(f"  {sys:20s}: {status}")

Anti-defence P/A matrix: (3460, 40)  (after sparsity filter)
Anti-defence systems dropped: 0

DEFENCE FEATURE BLOCK SUMMARY
Genomes (rows)         : 3460
Defence systems (cols) : 360
Anti-defence (cols)    : 40
Total feature columns  : 400

  SspBCDE             : PRESENT (311/900 genomes)
  Gao_Qat             : PRESENT (271/900 genomes)
  RM_Type_I           : PRESENT (2265/900 genomes)
  RM_Type_II          : PRESENT (1050/900 genomes)
  RM_Type_IV          : PRESENT (1508/900 genomes)


## Section 4  -  Parse ResFinder outputs (ARG counts)

In [15]:
arg_records = []   # one dict per genome

for sp in SPECIES:
    res_dir = INTERIM / sp / "resfinder"   # each genome has its own subdirectory

    for genome_dir in sorted(res_dir.iterdir()):
        if not genome_dir.is_dir():
            continue  # skip .DS_Store

        results_file = genome_dir / "ResFinder_results_tab.txt"

        if not results_file.exists():
            # ResFinder was called but produced no output  -  treat as zero ARGs
            arg_records.append({
                "genome_id": norm_acc(genome_dir.name),
                "species": sp,
                "arg_count_unique": 0,
                "arg_count_total": 0,
            })
            continue

        raw = pd.read_csv(results_file, sep="\t")

        # Some versions use slightly different headers; guard with fallback
        if "Resistance gene" in raw.columns:
            gene_col = "Resistance gene"
        else:
            gene_col = raw.columns[0]

        genome_id = norm_acc(genome_dir.name)

        if raw.empty:
            arg_records.append({
                "genome_id": genome_id, "species": sp,
                "arg_count_unique": 0, "arg_count_total": 0,
            })
            continue

        arg_records.append({
            "genome_id": genome_id,
            "species":   sp,
            "arg_count_unique": raw[gene_col].nunique(),  # distinct gene names
            "arg_count_total":  len(raw),                  # all hits including duplicates
        })

arg_df = pd.DataFrame(arg_records).set_index("genome_id")

print(f"ARG records parsed: {len(arg_df):,}  (expect 900)")
print(f"Genomes with >=1 ARG: {(arg_df['arg_count_unique'] > 0).sum()}")
print(f"Genomes with zero ARGs: {(arg_df['arg_count_unique'] == 0).sum()}")
print()
print("arg_count_unique distribution:")
print(arg_df["arg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median ARG count (unique):")
print(arg_df.groupby("species")["arg_count_unique"].median().sort_values(ascending=False).to_string())

ARG records parsed: 3,460  (expect 900)
Genomes with >=1 ARG: 3363
Genomes with zero ARGs: 97

arg_count_unique distribution:
count    3460.0
mean        8.7
std         6.3
min         0.0
25%         4.0
50%         7.0
75%        12.0
max        37.0

Per-species median ARG count (unique):
species
kpneumoniae    16.0
abaumannii     10.0
efaecium        8.0
ecloaceae       6.0
paeruginosa     6.0
saureus         2.0


## Section 5  -  Parse ICEberg outputs (IME/ICE counts)

Coverage filter uses max protein length per element as denominator (conservative direction: may miss some genuine hits but suppresses false positives).

In [16]:
from collections import defaultdict

def parse_fasta_max_lengths(fasta_path: Path) -> dict:
    """
    Parse a protein FASTA and return {first_word_of_header: max_sequence_length_aa}.
    When multiple sequences share the same first header word (as in ICEberg),
    the maximum length is kept  -  used as the conservative coverage denominator.
    """
    lengths = defaultdict(list)
    current_id = None
    current_len = 0
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    lengths[current_id].append(current_len)
                current_id = line[1:].split()[0]   # first word after '>'
                current_len = 0
            else:
                current_len += len(line)           # amino acid characters
    if current_id is not None:
        lengths[current_id].append(current_len)
    return {k: max(v) for k, v in lengths.items()}


# ── Build ICEberg protein length lookup ──────────────────────────────────────
ICEBERG_FASTA = ROOT / "data" / "raw" / "databases" / "ICEberg_IME.fasta"
ice_lengths = parse_fasta_max_lengths(ICEBERG_FASTA)

print(f"ICEberg unique element IDs with protein lengths: {len(ice_lengths)}")
# Spot-check: element 160 appears in the sample BLAST output
for eid in ["ICEberg|160_IME", "ICEberg|10_IME", "ICEberg|1_ICE"]:
    print(f"  {eid}: max protein length = {ice_lengths.get(eid, 'NOT FOUND')} aa")

ICEberg unique element IDs with protein lengths: 98
  ICEberg|160_IME: max protein length = 1072 aa
  ICEberg|10_IME: max protein length = 559 aa
  ICEberg|1_ICE: max protein length = NOT FOUND aa


In [17]:
BLAST_COLS = ["qseqid", "sseqid", "pident", "length", "mismatch",
              "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

PIDENT_MIN    = 40.0   # % identity threshold
COVERAGE_MIN  = 0.80   # 80% query coverage

ime_records = []

for sp in SPECIES:
    ice_dir = INTERIM / sp / "iceberg"

    for tsv_file in sorted(ice_dir.iterdir()):
        if not tsv_file.name.endswith("_iceberg.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_iceberg")
        genome_id = norm_acc(stem)

        # Empty file = zero ICEberg hits (legitimate result)
        if tsv_file.stat().st_size == 0:
            ime_records.append({
                "genome_id": genome_id, "species": sp,
                "ime_count_unique": 0, "ime_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # ── Apply filters ────────────────────────────────────────────────────
        raw = raw[raw["pident"] >= PIDENT_MIN]

        raw = raw[raw["qseqid"].isin(ice_lengths)]
        raw = raw.copy()
        raw["max_prot_len"] = raw["qseqid"].map(ice_lengths)
        raw["coverage"]     = (raw["qend"] - raw["qstart"] + 1) / raw["max_prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        ime_records.append({
            "genome_id":       genome_id,
            "species":         sp,
            "ime_count_unique": raw["qseqid"].nunique(),  # distinct element IDs
            "ime_count_total":  len(raw),                  # all passing hits
        })

ime_df = pd.DataFrame(ime_records).set_index("genome_id")

print(f"IME records: {len(ime_df):,}  (expect 900)")
print(f"Genomes with >=1 IME hit: {(ime_df['ime_count_unique'] > 0).sum()}")
print(f"Genomes with zero IMEs:   {(ime_df['ime_count_unique'] == 0).sum()}")
print()
print("ime_count_unique distribution:")
print(ime_df["ime_count_unique"].describe().round(1).to_string())
print()
print("Per-species median IME count (unique):")
print(ime_df.groupby("species")["ime_count_unique"].median().sort_values(ascending=False).to_string())

IME records: 3,460  (expect 900)
Genomes with >=1 IME hit: 3128
Genomes with zero IMEs:   332

ime_count_unique distribution:
count    3460.0
mean        9.9
std         6.9
min         0.0
25%         4.0
50%        11.0
75%        15.0
max        46.0

Per-species median IME count (unique):
species
kpneumoniae    17.0
ecloaceae      14.5
efaecium       12.0
abaumannii     11.0
saureus         4.0
paeruginosa     2.0


## Section 6  -  Parse BacMet outputs (HMRG counts) [SUPERSEDED  -  see Section 6b]

**This section is retained as a historical record only.** `hmrg_df` produced below is NOT included in the Section 9 join. BacMet was removed due to gram-stain reference bias  -  see `docs/decisions.md` (2026-05-12) and Section 6b.

In [18]:
BACMET_FASTA = ROOT / "data" / "raw" / "databases" / "BacMet2_EXP_database.fasta"

bacmet_lengths = parse_fasta_max_lengths(BACMET_FASTA)

print(f"BacMet unique protein IDs: {len(bacmet_lengths)}")
for pid in ["BAC0001|abeM|tr|Q5FAM9|Q5FAM9_ACIBA",
            "BAC0002|abeS|tr|Q2FD83|Q2FD83_ACIBA",
            "BAC0005|acrA|sp|P0AE06|ACRA_ECOLI"]:
    print(f"  {pid[:45]}: {bacmet_lengths.get(pid, 'NOT FOUND')} aa")

print()

hmrg_records = []

for sp in SPECIES:
    bm_dir = INTERIM / sp / "bacmet"

    for tsv_file in sorted(bm_dir.iterdir()):
        if not tsv_file.name.endswith("_bacmet.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_bacmet")
        genome_id = norm_acc(stem)

        if tsv_file.stat().st_size == 0:
            hmrg_records.append({
                "genome_id": genome_id, "species": sp,
                "hmrg_count_unique": 0, "hmrg_count_total": 0,
            })
            continue

        raw = pd.read_csv(tsv_file, sep="\t", header=None, names=BLAST_COLS)

        # pident filter
        raw = raw[raw["pident"] >= PIDENT_MIN]

        # Coverage filter  -  exact for BacMet (one protein per qseqid)
        raw = raw[raw["qseqid"].isin(bacmet_lengths)]
        raw = raw.copy()
        raw["prot_len"]  = raw["qseqid"].map(bacmet_lengths)
        raw["coverage"]  = (raw["qend"] - raw["qstart"] + 1) / raw["prot_len"]
        raw = raw[raw["coverage"] >= COVERAGE_MIN]

        hmrg_records.append({
            "genome_id":        genome_id,
            "species":          sp,
            "hmrg_count_unique": raw["qseqid"].nunique(),
            "hmrg_count_total":  len(raw),
        })

hmrg_df = pd.DataFrame(hmrg_records).set_index("genome_id")

print(f"HMRG records: {len(hmrg_df):,}  (expect 900)")
print(f"Genomes with >=1 HMRG: {(hmrg_df['hmrg_count_unique'] > 0).sum()}")
print(f"Genomes with zero HMRGs: {(hmrg_df['hmrg_count_unique'] == 0).sum()}")
print()
print("hmrg_count_unique distribution:")
print(hmrg_df["hmrg_count_unique"].describe().round(1).to_string())
print()
print("Per-species median HMRG count (unique):")
print(hmrg_df.groupby("species")["hmrg_count_unique"].median().sort_values(ascending=False).to_string())

BacMet unique protein IDs: 753
  BAC0001|abeM|tr|Q5FAM9|Q5FAM9_ACIBA: 448 aa
  BAC0002|abeS|tr|Q2FD83|Q2FD83_ACIBA: 109 aa
  BAC0005|acrA|sp|P0AE06|ACRA_ECOLI: 397 aa



HMRG records: 3,460  (expect 900)
Genomes with >=1 HMRG: 3460
Genomes with zero HMRGs: 0

hmrg_count_unique distribution:
count    3460.0
mean      204.9
std       113.4
min        39.0
25%        63.0
50%       231.0
75%       305.2
max       402.0

Per-species median HMRG count (unique):
species
kpneumoniae    332.0
ecloaceae      312.0
paeruginosa    278.5
abaumannii     177.0
saureus         60.0
efaecium        55.0


## Section 6b  -  AMRFinderPlus metal resistance genes

`--plus` unlocks the STRESS/METAL category; `--organism` loads species-specific HMM profiles (SA/EF copper and cadmium genes are absent from the pan-bacterial core database without it).

Filter: `Subtype == "METAL"`.

**Class values observed:**

| Class | Hits |
|---|---|
| MERCURY | 1684 |
| COPPER/SILVER | 1349 |
| COPPER | 1231 |
| ARSENIC | 1092 |
| TELLURIUM | 358 |
| SILVER | 354 |
| NA | 348 |
| NICKEL | 230 |
| CADMIUM | 56 |
| COPPER/NICKEL | 34 |
| CADMIUM/LEAD/ZINC | 5 |
| CHROMATE | 4 |

**Caveat: 82 crash genomes coded as zero.** `amr_report` v4.2.7 crashes on specific protein sequences; a 3-tier fallback recovered zero for all 82 affected genomes (SA 53/150, EF 28/150, AB 1/150). Coded as zero in the feature matrix.

In [19]:
# ── Metal class values observed in the actual AMRFinderPlus data ──────────────
METAL_CLASSES = [
    "MERCURY", "ARSENIC",
    "COPPER", "SILVER", "COPPER/SILVER",      # pco/sil variants
    "TELLURIUM",
    "NICKEL", "CADMIUM",
    "COPPER/NICKEL", "CADMIUM/LEAD/ZINC", "CHROMATE",
]

def metal_col(cls: str) -> str:
    """COPPER/SILVER → hmrg_copper_silver  (lowercase, / → _)"""
    return "hmrg_" + cls.lower().replace("/", "_")

amr_records = []

for sp in SPECIES:
    amr_dir = INTERIM / sp / "amrfinderplus"

    for tsv_file in sorted(amr_dir.iterdir()):
        if not tsv_file.name.endswith("_amrfinder.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_amrfinder")
        genome_id = norm_acc(stem)

        record = {"genome_id": genome_id, "species": sp}
        for cls in METAL_CLASSES:
            record[metal_col(cls)] = 0
        record["hmrg_unclassified"] = 0   # Class == "NA" rows
        record["hmrg_metal_total"]  = 0
        record["hmrg_metal_classes"] = 0

        # Header-only file = amr_report crash fallback (82 genomes)  -  treat as zero
        if tsv_file.stat().st_size == 0:
            amr_records.append(record)
            continue

        try:
            df_amr = pd.read_csv(tsv_file, sep="\t")
        except Exception:
            amr_records.append(record)
            continue

        if "Subtype" not in df_amr.columns or df_amr.empty:
            # 0-row DataFrame = header-only crash fallback
            amr_records.append(record)
            continue

        metal = df_amr[df_amr["Subtype"] == "METAL"].copy()

        if metal.empty:
            amr_records.append(record)
            continue

        record["hmrg_metal_total"] = len(metal)

        class_counts = metal["Class"].value_counts()
        for cls in METAL_CLASSES:
            record[metal_col(cls)] = int(class_counts.get(cls, 0))
        record["hmrg_unclassified"] = int(class_counts.get("NA", 0))

        # Metal class diversity: distinct Class values, NA excluded
        record["hmrg_metal_classes"] = int(
            metal[metal["Class"] != "NA"]["Class"].nunique()
        )

        amr_records.append(record)

amrfinder_df = pd.DataFrame(amr_records).set_index("genome_id")

print(f"AMRFinderPlus records: {len(amrfinder_df):,}  (expect 900)")
print(f"Genomes with >=1 METAL gene:              {(amrfinder_df['hmrg_metal_total'] > 0).sum()}")
print(f"Genomes coded as zero (crash or genuine): {(amrfinder_df['hmrg_metal_total'] == 0).sum()}")
print()
print("hmrg_metal_total distribution:")
print(amrfinder_df["hmrg_metal_total"].describe().round(1).to_string())
print()
print("Per-species median metal gene count:")
print(amrfinder_df.groupby("species")["hmrg_metal_total"].median()
      .sort_values(ascending=False).to_string())
print()
print("Per-species median metal class diversity (hmrg_metal_classes):")
print(amrfinder_df.groupby("species")["hmrg_metal_classes"].median()
      .sort_values(ascending=False).to_string())
print()
print("Metal class distribution (total hits across all 900 genomes):")
metal_col_list = [metal_col(cls) for cls in METAL_CLASSES]
print(amrfinder_df[metal_col_list].sum().sort_values(ascending=False).to_string())

AMRFinderPlus records: 3,460  (expect 900)
Genomes with >=1 METAL gene:              2859
Genomes coded as zero (crash or genuine): 601

hmrg_metal_total distribution:
count    3460.0
mean        7.5
std        11.1
min         0.0
25%         1.0
50%         2.0
75%        10.0
max        67.0

Per-species median metal gene count:
species
ecloaceae      21.0
kpneumoniae    20.0
abaumannii      1.0
efaecium        1.0
paeruginosa     1.0
saureus         1.0

Per-species median metal class diversity (hmrg_metal_classes):
species
ecloaceae      4.0
kpneumoniae    4.0
abaumannii     1.0
efaecium       1.0
paeruginosa    1.0
saureus        1.0

Metal class distribution (total hits across all 900 genomes):
hmrg_mercury              6201
hmrg_copper_silver        5216
hmrg_copper               4984
hmrg_arsenic              4243
hmrg_silver               1412
hmrg_tellurium            1316
hmrg_nickel                900
hmrg_cadmium               184
hmrg_copper_nickel         132
hmrg_chrom

## Section 7  -  ISEScan (IS element counts by family)

File: `data/interim/{sp}/isescan/{acc_fn}_isescan.tsv`  -  one row per IS element call: `seqID`, `family`, `cluster`, `isBegin`, `isEnd`, `isLen`, `type` (c = complete, p = partial).

32 empty files across genomes are legitimate zero-IS assemblies.

Output: `is_df` — index = genome_id, columns = `is_count_total` + `is_{family}_count` for each IS family observed.

In [20]:
is_record_list = []

for sp in SPECIES:
    is_dir = INTERIM / sp / "isescan"

    for tsv_file in sorted(is_dir.iterdir()):
        if not tsv_file.name.endswith("_isescan.tsv"):
            continue

        stem      = tsv_file.stem.removesuffix("_isescan")
        genome_id = norm_acc(stem)

        # Empty file = zero IS elements (~32 genomes  -  legitimate)
        if tsv_file.stat().st_size == 0:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        try:
            df_is = pd.read_csv(tsv_file, sep="\t")
        except Exception:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        if df_is.empty or "family" not in df_is.columns:
            is_record_list.append({
                "genome_id": genome_id, "species": sp, "is_count_total": 0
            })
            continue

        # Count IS elements per family for this genome
        family_counts = df_is["family"].value_counts()
        record = {
            "genome_id":      genome_id,
            "species":        sp,
            "is_count_total": len(df_is),   # all IS elements (complete + partial)
        }
        for fam, cnt in family_counts.items():
            record[f"is_{fam}_count"] = int(cnt)   # e.g. is_IS3_count, is_new_count

        is_record_list.append(record)

# fillna(0): families absent from a given genome get count 0
is_df = (
    pd.DataFrame(is_record_list)
    .fillna(0)
    .set_index("genome_id")
)

# Convert all IS count columns to int (fillna produces float)
is_count_cols = [c for c in is_df.columns if c.startswith("is_") and c.endswith("_count")]
is_df[is_count_cols + ["is_count_total"]] = (
    is_df[is_count_cols + ["is_count_total"]].astype(int)
)

is_families = sorted(c for c in is_df.columns if c.startswith("is_") and c != "is_count_total")

print(f"IS element records: {len(is_df):,}  (expect 900)")
print(f"Genomes with >=1 IS element:   {(is_df['is_count_total'] > 0).sum()}")
print(f"Genomes with zero IS elements: {(is_df['is_count_total'] == 0).sum()}  (expect ~32)")
print(f"IS families tracked: {len(is_families)}")
print()
print("is_count_total distribution:")
print(is_df["is_count_total"].describe().round(1).to_string())
print()
print("Per-species median IS element count:")
print(is_df.groupby("species")["is_count_total"].median()
      .sort_values(ascending=False).to_string())
print()
print("Top 10 IS families by total element count across all 900 genomes:")
family_totals = is_df[is_families].sum().sort_values(ascending=False)
print(family_totals.head(10).to_string())

IS element records: 1,131  (expect 900)
Genomes with >=1 IS element:   1082
Genomes with zero IS elements: 49  (expect ~32)
IS families tracked: 25

is_count_total distribution:
count    1131.0
mean       53.1
std        50.7
min         0.0
25%        19.0
50%        38.0
75%        64.0
max       297.0

Per-species median IS element count:
species
efaecium       154.0
kpneumoniae     60.0
ecloaceae       42.0
abaumannii      37.0
saureus         22.0
paeruginosa     20.0

Top 10 IS families by total element count across all 900 genomes:
is_IS3_count       10641
is_ISL3_count       7102
is_IS6_count        5491
is_IS5_count        5306
is_IS30_count       4840
is_IS256_count      4504
is_IS4_count        3127
is_IS21_count       2651
is_IS110_count      2616
is_IS1182_count     2216


## Section 8  -  MLST and metadata

In [21]:
# ── Expected MLST scheme per species ────────────────────────────────────────
EXPECTED_SCHEME = {
    "abaumannii":  "abaumannii_2",
    "ecloaceae":   "ecloacae",       # E. cloacae complex all use this scheme
    "efaecium":    "efaecium",
    "kpneumoniae": "klebsiella",
    "paeruginosa": "paeruginosa",
    "saureus":     "saureus",
}

mlst_records = []

for sp in SPECIES:
    mlst_file = INTERIM / sp / "mlst" / "mlst_results.tsv"

    # No header; columns: filepath \t scheme \t ST \t allele1 \t allele2 ...
    df_raw = pd.read_csv(
        mlst_file, sep="\t", header=None,
        usecols=[0, 1, 2],
        names=["filepath", "scheme", "st_raw"],
        dtype=str,
    )

    for _, row in df_raw.iterrows():
        genome_id = norm_acc(Path(row["filepath"]).stem)   # stem = GCF_XXX_1, norm_acc → GCF_XXX.1
        scheme    = row["scheme"].strip()
        st_raw    = row["st_raw"].strip()

        # Parse ST: "-" = untypeable; any "?" or "~" in the value = novel ST → NaN
        if st_raw == "-" or "?" in st_raw or "~" in st_raw:
            sequence_type = pd.NA
        else:
            try:
                sequence_type = int(st_raw)
            except ValueError:
                sequence_type = pd.NA

        # scheme "-" means typing failed entirely (genome issue, not species mismatch)
        scheme_mismatch = (scheme != EXPECTED_SCHEME[sp]) and (scheme != "-")

        mlst_records.append({
            "genome_id":       genome_id,
            "species":         sp,
            "mlst_scheme":     scheme,
            "sequence_type":   sequence_type,
            "scheme_mismatch": scheme_mismatch,
        })

mlst_df = (
    pd.DataFrame(mlst_records)
    .set_index("genome_id")
    .astype({"species": "category", "mlst_scheme": "category", "scheme_mismatch": bool})
)

# ── QC report ─────────────────────────────────────────────────────────────
print("── MLST scheme distribution per species ──")
print(mlst_df.groupby(["species", "mlst_scheme"]).size().rename("count").to_string())
print()
n_mismatch = int(mlst_df["scheme_mismatch"].sum())
print(f"Scheme mismatches: {n_mismatch} / {len(mlst_df)}")
print()
print(mlst_df[mlst_df["scheme_mismatch"]][["species", "mlst_scheme", "sequence_type"]])

── MLST scheme distribution per species ──
species      mlst_scheme    
abaumannii   abaumannii_2       600
ecloaceae    -                    2
             cronobacter         28
             ecloacae           505
             salmonella           1
efaecium     efaecium           524
kpneumoniae  ecoli_achtman_4     96
             klebsiella         504
paeruginosa  paeruginosa        600
saureus      saureus            600

Scheme mismatches: 125 / 3460

                     species      mlst_scheme sequence_type
genome_id                                                  
GCF_002237465.1    ecloaceae       salmonella          3011
GCF_006385655.1    ecloaceae      cronobacter          <NA>
GCF_008931525.1    ecloaceae      cronobacter           817
GCF_008931785.1    ecloaceae      cronobacter           419
GCF_012562255.1    ecloaceae      cronobacter           421
...                      ...              ...           ...
GCF_056113445.1  kpneumoniae  ecoli_achtman_4         11

### QC finding: 22 genomes carry wrong-species MLST scheme

| Species | Expected | Observed | Count |
|---------|----------|----------|-------|
| kpneumoniae | klebsiella | ecoli_achtman_4 | 18 |
| ecloaceae | ecloacae | cronobacter | 4 |

These 22 genomes passed CheckM2 quality gates (≥95% complete, ≤5% contaminated) but their core housekeeping genes type as a different genus. They are almost certainly species-misidentified NCBI submissions.

**They will be excluded in Section 9** at the final join step. The dataset drops from 900 to 878 genomes. This decision is logged in `docs/decisions.md`.

**Why CheckM2 didn't catch this:** CheckM2 tests completeness using a universal set of single-copy marker genes and measures contamination as the fraction of marker genes present in multiple copies. It does not test whether the marker genes match the expected species. A genome assembled entirely from *E. coli* DNA scores 100% complete and 0% contaminated by CheckM2  -  it looks like a perfect *E. coli* genome, which NCBI submitted as *K. pneumoniae*.

In [22]:
# ── Metadata: country, year_bin, complex_member ─────────────────────────────

meta_records = []

for sp in SPECIES:
    meta_file = INTERIM / sp / "metadata.tsv"
    df_raw = pd.read_csv(meta_file, sep="\t", dtype=str)

    for _, row in df_raw.iterrows():
        meta_records.append({
            "genome_id":      row["accession"],
            "species":        sp,
            "country":        row.get("country", "unknown"),
            "year_bin":       row.get("year_bin", "unknown"),
            "complex_member": row.get("complex_member", pd.NA),
        })

meta_df = (
    pd.DataFrame(meta_records)
    .set_index("genome_id")
    .astype({"species": "category"})
)

print(f"meta_df: {meta_df.shape[0]} genomes × {meta_df.shape[1]} columns")
print()
print("── Country distribution (top 10 across all species) ──")
print(meta_df["country"].value_counts().head(10).to_string())
print()
print("── Year bin distribution ──")
print(meta_df["year_bin"].value_counts().sort_index().to_string())
print()
print("── E. cloacae complex members ──")
print(meta_df[meta_df["species"] == "ecloaceae"]["complex_member"].value_counts().to_string())

meta_df: 3460 genomes × 4 columns

── Country distribution (top 10 across all species) ──
country
China             460
unknown           440
USA               439
South Korea       203
Taiwan            181
Germany           160
missing           159
United Kingdom    112
Australia         109
India             107

── Year bin distribution ──
year_bin
2010-2015        125
2016-2020       1194
2021-present    2134
pre-2010           7

── E. cloacae complex members ──
complex_member
E. hormaechei      265
E. roggenkampii     70
E. cloacae          66
E. asburiae         60
E. kobei            40
E. ludwigii         35


## Section 9  -  Join all blocks → feature matrix

1. Exclude 22 wrong-species MLST genomes.
2. IS sparsity filter; `IS200/IS605` sanitized to `IS200_IS605`.
3. Derived features: `defence_system_count`, `adef_system_count`, ratio features (denominator + 1 to prevent division by zero), `arg_burden_tertile` (within-species `pd.qcut(q=3)`).
4. Column prefixes: `dp_` = defence P/A, `dc_` = defence counts, `ad_` = anti-defence P/A.

In [23]:
# ── Step 1: identify included genomes ───────────────────────────────────────
mismatch_ids = set(mlst_df.index[mlst_df["scheme_mismatch"]])
included_ids  = [g for g in all_genome_ids if g not in mismatch_ids]

print(f"Excluded (scheme mismatch): {len(mismatch_ids)}")
print(f"Included:                   {len(included_ids)}")
print()
mismatch_by_sp = mlst_df[mlst_df["scheme_mismatch"]].groupby("species")["mlst_scheme"].value_counts()
print("Excluded breakdown:")
print(mismatch_by_sp.to_string())
print()

# ── Step 2: IS element sparsity filter ──────────────────────────────────────
# ISEScan is ~32% complete (~1,114 / 3,460 genomes).
# IS features excluded from this build to avoid false zeros for unrun genomes.
# Re-run this notebook after ISEScan completes to include IS columns.
is_df_clean = pd.DataFrame(index=included_ids)
print(f"IS features: EXCLUDED (ISEScan {len(is_df):,}/{len(all_genome_ids):,} genomes complete)")
print( "             is_df_clean is empty — IS columns will be added once ISEScan finishes.")

Excluded (scheme mismatch): 125
Included:                   3335

Excluded breakdown:
species      mlst_scheme    
ecloaceae    cronobacter        28
             salmonella          1
             paeruginosa         0
             klebsiella          0
             efaecium            0
             ecloacae            0
             saureus             0
             -                   0
             ecoli_achtman_4     0
             abaumannii_2        0
kpneumoniae  ecoli_achtman_4    96
             paeruginosa         0
             salmonella          0
             cronobacter         0
             -                   0
             abaumannii_2        0
             ecloacae            0
             efaecium            0
             klebsiella          0
             saureus             0

IS features: EXCLUDED (ISEScan 1,131/3,460 genomes complete)
             is_df_clean is empty — IS columns will be added once ISEScan finishes.


In [24]:
# ── Step 3: filter defence matrices to included genomes ─────────────────────
dp_mat = defence_pa.loc[included_ids]      # presence/absence
dc_mat = defence_count.loc[included_ids]   # copy counts
ad_mat = adef_pa.loc[included_ids]         # anti-defence presence/absence

# ── Summary features ─────────────────────────────────────────────────────────
defence_system_count = dp_mat.sum(axis=1).rename("defence_system_count")
adef_system_count    = ad_mat.sum(axis=1).rename("adef_system_count")

print("defence_system_count per species (median):")
sp_series = arg_df.loc[included_ids, "species"]
print(defence_system_count.groupby(sp_series).median()
      .sort_values(ascending=False).round(1).to_string())
print()

# ── Ratio features ────────────────────────────────────────────────────────────
# denominator +1 prevents division by zero (a handful of genomes have 0 defence systems)
arg_unique = arg_df.loc[included_ids, "arg_count_unique"]
ime_unique = ime_df.loc[included_ids, "ime_count_unique"]

ratio_arg_defence  = (arg_unique  / (defence_system_count + 1)).rename("ratio_arg_defence")
ratio_ime_defence  = (ime_unique  / (defence_system_count + 1)).rename("ratio_ime_defence")
ratio_adef_defence = (adef_system_count / (defence_system_count + 1)).rename("ratio_adef_defence")

# ── Q2 target: within-species ARG burden tertile ─────────────────────────────
tertile_labels = pd.Series(index=pd.Index(included_ids), dtype="object",
                            name="arg_burden_tertile")

for sp in SPECIES:
    sp_ids = sp_series[sp_series == sp].index
    sp_arg = arg_unique[sp_ids]
    try:
        labels = pd.qcut(
            sp_arg, q=3,
            labels=["low_ARG", "mid_ARG", "high_ARG"],
            duplicates="drop",
        )
    except ValueError:
        # Protocol Amendment PA-1: tertile failed (0th == 33rd percentile,
        # floor effect). Pre-specified fallback: binary split at the median.
        med = sp_arg.median()
        result_vals = np.where(sp_arg < med, "low_ARG",
                      np.where(sp_arg > med, "high_ARG", "mid_ARG"))
        labels = pd.Series(result_vals, index=sp_ids, dtype="object")
        print(f"  {sp}: floor effect → binary split at median={med:.0f} "
              f"| low={(sp_arg < med).sum()} "
              f"mid={(sp_arg == med).sum()} "
              f"high={(sp_arg > med).sum()}")
    tertile_labels[sp_ids] = labels.astype(str).values

print("ARG burden tertile  -  counts per species:")
print(pd.crosstab(sp_series, tertile_labels).to_string())
print()
print("Overall tertile totals:")
print(tertile_labels.value_counts().to_string())

defence_system_count per species (median):
species
kpneumoniae    22.0
ecloaceae      17.0
paeruginosa    14.0
saureus        12.0
efaecium        8.0
abaumannii      6.0

ARG burden tertile  -  counts per species:
arg_burden_tertile  high_ARG  low_ARG  mid_ARG
species                                       
abaumannii               189      234      177
ecloaceae                155      217      135
efaecium                 160      205      159
kpneumoniae              153      178      173
paeruginosa              196      275      129
saureus                  195      254      151

Overall tertile totals:
arg_burden_tertile
low_ARG     1363
high_ARG    1048
mid_ARG      924


### Protocol Amendment PA-1: P. aeruginosa uses binary split for Q2

#### What went wrong with the tertile split

`pd.qcut(q=3, duplicates='drop')` on PA `arg_count_unique` produces bin edges **[5.0, 5.0, 8.0, 29.0]**  -  the 0th and 33rd percentiles are both 5.0 because **56/150 PA genomes (37%)** sit at ARG = 5, the species minimum. After `duplicates='drop'`, only 2 unique edges survive → 1 bin → `ValueError: Bin labels must be one fewer than the number of bin edges`.

#### Why PA has a floor at ARG = 5

P. aeruginosa clinical isolates almost universally carry a small set of near-baseline acquired ARGs. ResFinder captures these as 5 unique entries per genome. Chromosomal cephalosporinase derivatives (blaPDC) and housekeeping efflux pump genes are ubiquitous in clinical PA, creating a hard floor. The right-skewed tail (PA genomes reaching ARG = 29) represents those that have acquired *additional* ARGs via MGEs  -  exactly the genomes where the RESTRICT/FACILITATE dichotomy should be testable.

#### Three options evaluated

| Option | Description | Verdict |
|--------|-------------|---------|
| 1. Exclude PA | All PA genomes → mid_ARG; PA out of Q2 | Rejected: loses 64 above-median PA genomes and all cross-species PA signal |
| **2. Binary split at median** | Below median → low_ARG; above → high_ARG; at median → mid_ARG | **Chosen**  -  biologically coherent, weaker but valid |
| 3. Rank-based tertile | Split by rank order into thirds | Rejected: 56 genomes share ARG=5 exactly; different labels for identical values = noise as signal |

#### Why binary split is biologically valid

Q2 for PA asks: do PA genomes at the ARG floor (ARG < 6) differ in defence profile from those with above-average burden (ARG > 6)? RM systems should be enriched in the low-ARG group; permissive systems (SspBCDE, Gao_Qat) in the high-ARG group  -  if the cross-species RESTRICT/FACILITATE pattern holds for PA. This is a coherent test of the core hypothesis, slightly weaker than the tertile contrast but not arbitrary.

#### Why this is a valid amendment, not post-hoc modification

1. **Timing:** Applied in Phase 3 (feature engineering) before any model performance is seen. No Q2 accuracy numbers for any species have been computed.
2. **Criterion is data-structural:** The fallback condition (0th percentile == 33rd percentile) is evaluated from the ARG distribution alone, independently of ML results.
3. **Generalises to future datasets:** The rule applies to any species with a floor effect, regardless of which species. Not PA-specific.
4. **Documented prospectively in `docs/pre_analysis_plan.md` §7 (Amendment PA-1).**

#### Result after amendment

PA: `low_ARG` = **56 genomes** (ARG < 6) | `mid_ARG` = **30 genomes** (ARG = 6, excluded from Q2) | `high_ARG` = **64 genomes** (ARG > 6).

Q2 now covers all **6 ESKAPE species**, **778 eligible genomes** (was 5 species, 728 genomes before this amendment).

In [25]:
# ── Step 4: assemble feature matrix ─────────────────────────────────────────
feature_matrix = pd.concat(
    [
        # Defence system blocks  -  add prefixes to make feature type explicit
        dp_mat.rename(columns=lambda c: f"dp_{c}"),
        dc_mat.rename(columns=lambda c: f"dc_{c}"),
        ad_mat.rename(columns=lambda c: f"ad_{c}"),

        # Per-tool count blocks
        arg_df.loc[included_ids,      ["arg_count_unique", "arg_count_total"]],
        ime_df.loc[included_ids,      ["ime_count_unique", "ime_count_total"]],
        amrfinder_df.loc[included_ids].drop(columns=["species"]),
        is_df_clean,

        # Summary and ratio features
        defence_system_count,
        adef_system_count,
        ratio_arg_defence,
        ratio_ime_defence,
        ratio_adef_defence,

        # Labels and metadata (not ML input  -  used for target and stratification)
        sp_series.rename("species"),
        tertile_labels,
        meta_df.loc[included_ids,  ["country", "year_bin", "complex_member"]],
        mlst_df.loc[included_ids,  ["sequence_type", "mlst_scheme"]],
    ],
    axis=1,
)

# ── Validation ───────────────────────────────────────────────────────────────
assert feature_matrix.shape[0] == len(included_ids), "Row count mismatch"
assert feature_matrix.index.duplicated().sum() == 0, "Duplicate genome IDs"

n_missing = feature_matrix.isnull().sum()
missing_cols = n_missing[n_missing > 0]

feature_cols = [c for c in feature_matrix.columns
                if c.startswith(("dp_", "dc_", "ad_", "arg_", "ime_",
                                  "hmrg_", "is_", "defence_", "adef_", "ratio_"))]
label_cols   = ["species", "arg_burden_tertile", "country", "year_bin",
                "complex_member", "sequence_type", "mlst_scheme"]

print(f"Feature matrix: {feature_matrix.shape[0]} genomes × {feature_matrix.shape[1]} columns")
print()
print(f"Feature columns: {len(feature_cols)}")
print(f"  dp_* (defence P/A):         {sum(c.startswith('dp_') for c in feature_cols)}")
print(f"  dc_* (defence counts):      {sum(c.startswith('dc_') for c in feature_cols)}")
print(f"  ad_* (anti-defence P/A):    {sum(c.startswith('ad_') for c in feature_cols)}")
print(f"  arg / ime / hmrg / is_:     {sum(c.startswith(('arg_','ime_','hmrg_','is_')) for c in feature_cols)}")
print(f"  summary + ratio:            {sum(c.startswith(('defence_','adef_','ratio_')) for c in feature_cols)}")
print(f"Label / metadata columns: {len(label_cols)}")
print()
if len(missing_cols) > 0:
    print("WARNING  -  missing values in:")
    print(missing_cols[missing_cols > 0].to_string())
else:
    print("Missing values in feature columns: none")
print()

# ── Save ─────────────────────────────────────────────────────────────────────
out_path = PROC / "feature_matrix_3460.parquet"
feature_matrix.to_parquet(out_path)

fm_check = pd.read_parquet(out_path)
assert fm_check.shape == feature_matrix.shape

print(f"Saved → {out_path}")
print(f"File size: {out_path.stat().st_size / 1_000_000:.1f} MB")
print("Round-trip check passed.")

Feature matrix: 3335 genomes × 790 columns

Feature columns: 784
  dp_* (defence P/A):         360
  dc_* (defence counts):      360
  ad_* (anti-defence P/A):    40
  arg / ime / hmrg / is_:     19
  summary + ratio:            5
Label / metadata columns: 7

WARNING  -  missing values in:
country           10
sequence_type    148



Saved → ../data/processed/feature_matrix_3460.parquet
File size: 0.7 MB
Round-trip check passed.
